# Executar produtor Kafka (`producer_alunos.py`)
Tech Challenge Fase 2 - Pipeline Híbrida de Alfabetização

Este notebook instala a dependência, configura as credenciais do Confluent Cloud via widgets e executa o script `producer_alunos.py` que você já enviou ao workspace.

**Antes de rodar:** faça upload de `producer_alunos.py` para o mesmo diretório deste notebook no Workspace (ou para o Volume/DBFS de sua preferência) e ajuste o widget `script_path` na célula 3 se necessário.

## 1. Instalar a biblioteca `confluent-kafka`

In [0]:
%pip install confluent-kafka --break-system-packages

In [0]:
# Reinicia o interpretador Python para que a biblioteca recém-instalada
# fique disponível na sessão atual
dbutils.library.restartPython()

## 2. Configurar caminho do script e credenciais
Preencha os widgets criados abaixo (ícone de controles no topo do notebook).

- `script_path`: caminho completo até o `producer_alunos.py` no workspace.
  Exemplos comuns: `/Workspace/Users/seu.email@empresa.com/producer_alunos.py` ou `/Workspace/Repos/seu-repo/producer_alunos.py`.

In [0]:
dbutils.widgets.text("script_path", "producer_alunos.py", "Caminho do script")
dbutils.widgets.text("bootstrap_servers", "", "Bootstrap Servers")
dbutils.widgets.text("api_key", "", "API Key")
dbutils.widgets.text("api_secret", "", "API Secret")
dbutils.widgets.text("topic", "alunos-eventos", "Kafka Topic")
dbutils.widgets.text("intervalo", "2", "Intervalo (s)")
dbutils.widgets.text("quantidade", "50", "Quantidade de eventos")

In [0]:
import os

os.environ["CONFLUENT_BOOTSTRAP_SERVERS"] = dbutils.widgets.get("bootstrap_servers")
os.environ["CONFLUENT_API_KEY"] = dbutils.widgets.get("api_key")
os.environ["CONFLUENT_API_SECRET"] = dbutils.widgets.get("api_secret")
os.environ["KAFKA_TOPIC"] = dbutils.widgets.get("topic")

faltando = [k for k in ("CONFLUENT_BOOTSTRAP_SERVERS", "CONFLUENT_API_KEY",
                        "CONFLUENT_API_SECRET") if not os.environ.get(k)]
if faltando:
    raise ValueError(f"Preencha os widgets: {', '.join(faltando)}")

script_path = dbutils.widgets.get("script_path")
if not os.path.exists(script_path):
    raise FileNotFoundError(
        f"Script não encontrado em '{script_path}'. "
        "Confira o widget 'script_path' e se o arquivo foi enviado ao workspace."
    )

print("Credenciais configuradas para o tópico:", os.environ["KAFKA_TOPIC"])
print("Script localizado em:", script_path)

## 3. Executar o produtor
Equivalente a rodar `python producer_alunos.py --intervalo 2 --quantidade 50` no terminal, mas passando o ambiente (`os.environ`) explicitamente para o subprocesso — assim as credenciais chegam ao script mesmo sem `export` persistir entre células.

In [0]:
import subprocess

comando = [
    "python", script_path,
    "--intervalo", dbutils.widgets.get("intervalo"),
    "--quantidade", dbutils.widgets.get("quantidade"),
]

processo = subprocess.Popen(
    comando,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for linha in processo.stdout:
    print(linha, end="")

processo.wait()
print(f"\nProcesso finalizado com código de saída {processo.returncode}")